In [1]:
"""REGULARIZATION"""

'REGULARIZATION'

In [10]:
# Import Necessary Libraries

import pandas as pd
import numpy as np

data_path = "banking_dataset.csv"
df = pd.read_csv(data_path)

print("Loaded Dataset:", data_path)
print("Shape:", df.shape)

Loaded Dataset: banking_dataset.csv
Shape: (1000, 31)


In [11]:
#Step 3: Define features and target

target_col = "bank_profit"

X = df.drop(columns =[target_col])

Y = df[target_col]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (1000, 30)
Y shape: (1000,)


In [12]:
# Step 4: Standardize and apply Linear Regression

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scale = scaler.fit_transform(X)
X_scale = pd.DataFrame(X_scale,columns = X.columns)

print("X_scale shape:", X_scale.shape)

X_scale shape: (1000, 30)


In [13]:
X_scale.head()

,customer_age,account_balance,annual_income,loan_amount,loan_duration_months,num_of_credit_cards,credit_card_utilization,monthly_salary,num_of_loans,num_of_defaults,...,house_value,insurance_amount,spending_score,digital_transactions_ratio,mobile_banking_usage_hours,branch_visits_per_month,internet_banking_logins,loan_interest_rate,customer_satisfaction_score,risk_factor
0,0.812959,-1.691764,1.742949,1.945064,0.642796,0.727699,1.189854,-0.317263,0.418966,1.258777,...,-0.460125,0.760323,1.643617,-0.750647,-0.873148,1.267172,-1.036244,-1.408255,0.513184,0.893274
1,1.680578,0.146953,-0.429639,-0.751237,-0.415392,-0.697764,0.043626,0.282552,1.300999,1.258777,...,0.906162,-0.909685,-1.723542,1.297783,-0.271305,0.550446,0.710234,1.123715,-1.178497,-1.327247
2,0.145560,-0.828137,0.208183,-0.539084,-1.367761,0.727699,0.113095,0.289409,-1.345101,-1.202175,...,1.684177,-0.179603,-1.320737,0.819816,-0.483986,1.625535,-1.152676,1.794282,-0.724728,1.032056
3,-0.788799,-1.504639,0.931397,0.837951,-1.473580,0.014967,0.217297,0.041627,-0.463068,1.258777,...,0.581518,0.443462,0.149552,1.434345,0.683499,-1.599732,-1.152676,-1.448721,-0.995397,-1.604812
4,1.079919,-0.716613,-1.470399,-0.380359,-0.521211,-1.410495,-1.137336,-0.615021,0.418966,0.028301,...,-0.666157,-0.319195,1.558852,0.614973,0.366740,-0.524643,-0.221221,-0.159612,0.186789,-0.182291


In [14]:
#Step 5: Split into train and test sets

from sklearn.model_selection import train_test_split, ShuffleSplit

X_train, X_test, Y_train, Y_test = train_test_split(X_scale, Y, test_size = 0.3, random_state = 42)

print("Train Size:", X_train.shape[0])
print("Test Size:", X_test.shape[0])

Train Size: 700
Test Size: 300


In [15]:
#Step 6: Modelfit and Evaluation

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr = LinearRegression()
lr.fit(X_train,Y_train)

#Evaluate Linear Regression

Y_train_pred_lr = lr.predict(X_train)
Y_test_pred_lr = lr.predict(X_test)

lr_train_r2 = r2_score(Y_train, Y_train_pred_lr)
lr_test_r2 = r2_score(Y_test, Y_test_pred_lr)

print("Linear Regression (single split)")
print(f"Train R^2 :{lr_train_r2:.4f}")
print(f"Test R^2 :{lr_test_r2:.4f}")

Linear Regression (single split)
Train R^2 :0.7243
Test R^2 :0.7136


In [16]:
# Step 7: Apply ShuffleSplit Cross-Validation

from sklearn.model_selection import ShuffleSplit, cross_validate

#cross-val-score

shuffle_split = ShuffleSplit(n_splits =500, test_size =0.2, random_state =42)

cv_scores = cross_validate(lr, X_scale, Y, cv=shuffle_split, scoring='r2', return_train_score= True)

#Extract train and test scores

train_scores = cv_scores['train_score']

test_scores = cv_scores['test_score']

# Show individual scores and their means

print("cross validation: training score:" , np.round(train_scores.mean(),2))
print("cross validation: test score:" , np.round(test_scores.mean(),2))

cross validation: training score: 0.73
cross validation: test score: 0.7


In [17]:
# Step 8: Apply LassCV to find the best alpha

In [18]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import ShuffleSplit

import numpy as np

#Define the ShuffleSplit Cross- Validation strategy

shuffle_split_lassocv = ShuffleSplit(
    n_splits = 100,
    test_size = 0.2,
    random_state = 42
)

In [19]:
# Define the LassoCV model
# The 'alphas' parameter can be left as Name to let LassoCV automatically determine the path.

lassocv_model = LassoCV(
    cv = shuffle_split_lassocv,
    random_state = 42
)
# Fit the model
lassocv_model.fit(X_scale,Y)

# Get the best alpha and corresponding mean_squared_error.

best_alpha_lassocv = lassocv_model.alpha_        # Best alpha selected based on lowest error
mse_path = lassocv_model.mse_path_               # Stores MSE values for each alpha across all splits
mean_mse = mse_path.mean(axis = 1)               # Compute average MSE for each alpha over 100 splits

best_mse = mean_mse[lassocv_model.alphas_ == best_alpha_lassocv] [0]      # Extract the MSE corresponding to the best alpha.

print("\n Lasso CV Results:")
print("Best alpha found:", best_alpha_lassocv)
print("Mean Squared Error for best alpha:", best_mse)           # You can also access the mean squared error for best alpha


 Lasso CV Results:
Best alpha found: 2651.425058310539
Mean Squared Error for best alpha: 2502729312.717604


In [20]:
#Step 11: Fit Lasso model with best alpha and count zero / non-zero coefficients.

from sklearn.linear_model import Lasso           # Import Lasso Regression model.

final_lasso_model = Lasso(alpha = best_alpha_lassocv)          # Create Lasso model using best alpha from LassoCV

final_lasso_model.fit(X_scale, Y)         # Train model on entire dataset (X_scale = features, Y = target)
                                          # Using full data to analyze final coefficients.

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",np.float64(2651.425058310539)
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [21]:
# Get the Coefficients

lasso_coefficients = final_lasso_model.coef_              # Extract learned weights (importance of each feature)
zero_coefficients = np.sum(lasso_coefficients == 0)       # Count zero and non-zero coefficients
non_zero_coefficients = np.sum(lasso_coefficients != 0)

print("\nAnalysis of Lasso Coefficients (with best alpha):")
print("Number of Zero Coefficient:", zero_coefficients)
print("Number of Non-zero Coefficients", non_zero_coefficients)


Analysis of Lasso Coefficients (with best alpha):
Number of Zero Coefficient: 26
Number of Non-zero Coefficients 4


In [23]:
# Get the Indices of the Non-Zero Coefficients

non_zero_indices = np.nonzero(lasso_coefficients)[0]

In [24]:
# Get the Original Variable names of the selected features

selected_feature_names = X.columns[non_zero_indices]

print("\nNames of the Variable with Non Zero Coefficients")
print(selected_feature_names)


Names of the Variable with Non Zero Coefficients
Index(['annual_income', 'loan_amount', 'num_of_defaults', 'investment_value'], dtype='str')


In [25]:
X_final = X_scale[selected_feature_names]

In [26]:
X_final.head()

,annual_income,loan_amount,num_of_defaults,investment_value
0,1.742949,1.945064,1.258777,2.017967
1,-0.429639,-0.751237,1.258777,2.744418
2,0.208183,-0.539084,-1.202175,-0.221609
3,0.931397,0.837951,1.258777,0.128706
4,-1.470399,-0.380359,0.028301,0.669426


In [27]:
#Step 12: Apply ShuffleSplit and Cross Validation

from sklearn.model_selection import ShuffleSplit, cross_validate

#Cross_VAl_Score

shuffle_split = ShuffleSplit(
    n_splits = 500,
    test_size = 0.2,
    random_state = 42
)

cv_scores = cross_validate(
    lr,
    X_final,
    Y,
    cv=shuffle_split,
    scoring ='r2',
return_train_score = True
)

In [28]:
#Extract Train and Test Scores

train_scores = cv_scores['train_score']
test_scores = cv_scores['test_score']

In [29]:
#Show Individual scores and their means

print("cross validate: training score:", np.round(train_scores.mean(), 2))
print("cross Validate: test score:", np.round(test_scores.mean(), 2))

cross validate: training score: 0.72
cross Validate: test score: 0.71


In [31]:
# Find the best Alpha using LassoCV

from sklearn.linear_model import LassoCV
from sklearn.model_selection import ShuffleSplit

# cross_val_score

shuffle_split = ShuffleSplit(
    n_splits = 500,
    test_size = 0.2,
    random_state = 42
)

cv_scores = cross_validate(
    lr,
    X_final,
    Y,
    cv = shuffle_split,
    scoring = 'r2',
    return_train_score = True
)

In [32]:
# Extract Train and Test scores.

train_scores = cv_scores['train_score']

test_scores = cv_scores['test_score']

print("Cross Validation : training score:", np.round(train_scores.mean(), 2))

print("Cross Validation : testing score:", np.round(test_scores.mean(), 2))

Cross Validation : training score: 0.72
Cross Validation : testing score: 0.71


In [33]:
#Find best Alpha using LassoCV

from sklearn.linear_model import LassoCV
from sklearn.model_selection import ShuffleSplit

#Cross Validation Strategy

cv = ShuffleSplit(
    n_splits = 100,
    test_size = 0.2,
    random_state = 42
)

#LassoCV (L1 Regularization)

lasso_cv = LassoCV(
    cv = cv,
    random_state = 42
)
lasso_cv.fit(X_scale,Y)

best_alpha_l1 = lasso_cv.alpha_

print("Best Alpha(L1):", best_alpha_l1)

Best Alpha(L1): 2651.425058310539


In [34]:
#Train Final Lasso Model

from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha = best_alpha_l1)
lasso_model.fit(X_scale,Y)

coeff_l1 = lasso_model.coef_

print((coeff_l1 != 0).sum())
print((coeff_l1 ==0).sum())

4
26


In [35]:
#Find best Alpha using RidgeCV

from sklearn.linear_model import RidgeCV
import numpy as np

ridge_cv = RidgeCV(
    alphas = np.logspace(-3, 3, 100),

    cv = cv
)

ridge_cv.fit(X_scale, Y)
best_alpha_l2 = ridge_cv.alpha_

print("Best Alpha(l2):", best_alpha_l2)

Best Alpha(l2): 26.560877829466893


In [36]:
#Train Final Ridge Model

from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha = best_alpha_l2)
ridge_model.fit(X_scale, Y)

coef_l2 = ridge_model.coef_

print("Ridge Coefficient:", coef_l2)

Ridge Coefficient: [ -762.60171545 -2185.28204057 72953.48696877 20384.01444461
 -1140.44187934  -518.72567971   394.65166322 -1017.19065031
  2010.58851447 -4965.14992048 -2182.01667007 -2371.87581707
 -1704.2326214    354.59091925  5458.87205073  -341.88027139
 -1846.56089565 -1639.74258349 -1856.79588385   759.26618908
   153.10905061   500.18456142   494.18282179 -1779.71863272
  -239.12811693  2382.08156893  1424.40534471   843.5897797
 -2034.28071253 -1988.17490227]


In [37]:
# Elastic Net (L1+L2)
#STEP 1: Find Best Alpha using ElasticNetCV

from sklearn.linear_model import ElasticNetCV

elastic_cv = ElasticNetCV(
    l1_ratio = [0.1, 0.5,0.7, 0.9],

    cv = cv,
    random_state = 42
)

elastic_cv.fit(X_scale, Y)
best_alpha_en = elastic_cv.alpha_            # Get best alpha (regularization strength)
                                             # This value gives lowest error.
best_l1_ratio = elastic_cv.l1_ratio_

print("Best Alpha (ElasticNet):", best_alpha_en)
print("Best l1 ratio:", best_l1_ratio)

l2_ratio = 1 - best_l1_ratio

print("L2 Ratio:", l2_ratio)

Best Alpha (ElasticNet): 83.90392965042108
Best l1 ratio: 0.9
L2 Ratio: 0.09999999999999998


In [38]:
#Train Final ElasticNet Model

from sklearn.linear_model import ElasticNet

elastic_model = ElasticNet(
    alpha = best_alpha_en,
    l1_ratio = best_l1_ratio
)

elastic_model.fit(X_scale, Y)
coeff_en = elastic_model.coef_

print("ElasticNet Coefficient :", coeff_en)
print("Zero Coefficient :", (coeff_en == 0).sum())
print("Non Zero Coefficient :",(coeff_en != 0).sum())

ElasticNet Coefficient : [ 2.70888301e+01 -5.13401946e+02  8.00403658e+03  2.38624633e+03
 -2.63141378e+02 -7.47335979e+01 -1.48498856e+02 -2.10067255e+02
 -7.82573395e+01 -6.55607470e+02 -5.37887310e+02  4.04491628e+02
 -1.13379951e+02 -1.71390052e+02  6.56726692e+02 -3.79634858e+02
 -5.63210355e+02 -2.97967222e+02 -2.91328164e+02 -1.15942412e+02
 -2.94478630e+02  3.15078690e+02 -4.98001497e+01 -3.12438986e+00
 -7.79629953e+01  2.49800070e+02 -8.95070762e+01  1.66192248e+02
  5.57825152e+00 -1.76679512e+02]
Zero Coefficient : 0
Non Zero Coefficient : 30
